# Experiment 5B: Test-Time Augmentation (TTA)

Evaluate the fresh baseline model with TTA to measure free mAP boost.
No training required — just `augment=True` during validation.

In [ ]:
# Cell 1: Setup
from ultralytics import YOLO
from pathlib import Path
import json
import time

ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
WEIGHTS = ROOT / "runs" / "detect" / "fresh_baseline" / "weights" / "best.pt"
EVALS_DIR = ROOT / "evals"

assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS}'
print(f'Weights: {WEIGHTS}')
print(f'Data: {DATA_YAML}')

## Baseline (No TTA) — Reference

In [ ]:
# Cell 2: Baseline evaluation (no TTA) for comparison
print("Running baseline evaluation (no TTA)...")
model = YOLO(str(WEIGHTS))

start = time.time()
baseline_results = model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    augment=False,
    verbose=True,
)
baseline_time = time.time() - start

baseline_map50 = float(baseline_results.box.map50)
baseline_map50_95 = float(baseline_results.box.map)
baseline_precision = float(baseline_results.box.mp)
baseline_recall = float(baseline_results.box.mr)

print()
print('=' * 60)
print('BASELINE (No TTA)')
print('=' * 60)
print(f'mAP@0.5:      {baseline_map50:.4f} ({baseline_map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {baseline_map50_95:.4f} ({baseline_map50_95*100:.1f}%)')
print(f'Precision:    {baseline_precision:.4f}')
print(f'Recall:       {baseline_recall:.4f}')
print(f'Time:         {baseline_time:.1f}s')
print('=' * 60)

## TTA Evaluation

In [ ]:
# Cell 3: TTA evaluation
print("Running TTA evaluation (augment=True)...")
model = YOLO(str(WEIGHTS))

start = time.time()
tta_results = model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    augment=True,
    verbose=True,
)
tta_time = time.time() - start

tta_map50 = float(tta_results.box.map50)
tta_map50_95 = float(tta_results.box.map)
tta_precision = float(tta_results.box.mp)
tta_recall = float(tta_results.box.mr)

print()
print('=' * 60)
print('TTA RESULTS')
print('=' * 60)
print(f'mAP@0.5:      {tta_map50:.4f} ({tta_map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {tta_map50_95:.4f} ({tta_map50_95*100:.1f}%)')
print(f'Precision:    {tta_precision:.4f}')
print(f'Recall:       {tta_recall:.4f}')
print(f'Time:         {tta_time:.1f}s')
print('=' * 60)

## Per-Class AP Comparison

In [ ]:
# Cell 4: Per-class comparison
class_names = baseline_results.names

print('=' * 70)
print('PER-CLASS AP@0.5 COMPARISON')
print('=' * 70)
header = f"{'Class':<20} {'Baseline':>10} {'TTA':>10} {'Diff':>10}"
print(header)
print('-' * 70)

per_class = {}
for cls_id, cls_name in class_names.items():
    bl_ap = float(baseline_results.box.ap50[cls_id]) if cls_id < len(baseline_results.box.ap50) else 0
    tta_ap = float(tta_results.box.ap50[cls_id]) if cls_id < len(tta_results.box.ap50) else 0
    diff = tta_ap - bl_ap
    per_class[cls_name] = {'baseline': bl_ap, 'tta': tta_ap, 'diff': diff}
    sign = '+' if diff > 0 else ''
    print(f"{cls_name:<20} {bl_ap*100:>9.1f}% {tta_ap*100:>9.1f}% {sign}{diff*100:>9.1f}%")

print('-' * 70)
bl_str = f'{baseline_map50*100:.1f}%'
tta_str = f'{tta_map50*100:.1f}%'
diff_str = f'{(tta_map50-baseline_map50)*100:+.1f}%'
print(f"{'mAP@0.5':<20} {bl_str:>10} {tta_str:>10} {diff_str:>10}")
print('=' * 70)

## FPS Calculation

In [ ]:
# Cell 5: FPS measurement
model = YOLO(str(WEIGHTS))
test_dir = ROOT / 'datasets' / 'NEU-DET' / 'yolo' / 'images' / 'test'
test_images = list(test_dir.glob('*.jpg'))
print(f'Test images: {len(test_images)}')

# Warmup
for img in test_images[:5]:
    model(str(img), verbose=False)

# FPS (no TTA)
start = time.time()
for img in test_images:
    model(str(img), verbose=False)
fps_no_tta = len(test_images) / (time.time() - start)

# FPS (with TTA)
start = time.time()
for img in test_images:
    model(str(img), verbose=False, augment=True)
fps_tta = len(test_images) / (time.time() - start)

print()
print('=' * 60)
print('FPS BENCHMARK')
print('=' * 60)
print(f'FPS (no TTA): {fps_no_tta:.1f}')
print(f'FPS (TTA):    {fps_tta:.1f}')
print(f'TTA overhead: {fps_no_tta/fps_tta:.1f}x slower')
print('=' * 60)

## Save Results

In [ ]:
# Cell 6: Save all metrics to JSON
metrics = {
    'experiment': '5B_tta_evaluation',
    'model': 'yolo11n',
    'weights': str(WEIGHTS),
    'baseline': {
        'map50': baseline_map50,
        'map50_95': baseline_map50_95,
        'precision': baseline_precision,
        'recall': baseline_recall,
        'eval_time_seconds': baseline_time,
    },
    'tta': {
        'map50': tta_map50,
        'map50_95': tta_map50_95,
        'precision': tta_precision,
        'recall': tta_recall,
        'eval_time_seconds': tta_time,
    },
    'improvement': {
        'map50_delta': tta_map50 - baseline_map50,
        'map50_95_delta': tta_map50_95 - baseline_map50_95,
        'precision_delta': tta_precision - baseline_precision,
        'recall_delta': tta_recall - baseline_recall,
    },
    'per_class_ap50': per_class,
    'fps': {
        'no_tta': fps_no_tta,
        'tta': fps_tta,
        'tta_overhead_x': fps_no_tta / fps_tta,
    },
}

out_path = EVALS_DIR / 'exp_5b_tta_results.json'
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Results saved to: {out_path}')

# Final summary
print()
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Baseline mAP@0.5: {baseline_map50*100:.1f}%')
print(f'TTA mAP@0.5:      {tta_map50*100:.1f}%')
print(f'Improvement:      {(tta_map50-baseline_map50)*100:+.1f}%')
if tta_map50 >= 0.80:
    print()
    print('*** 80% TARGET REACHED! ***')
else:
    gap = (0.80 - tta_map50) * 100
    print()
    print(f'Gap to 80%: {gap:.1f}%')
print('=' * 60)